# SDG 15.3.1 / UNCCD Reporting on Google Earth Engine

**Methodology reference:** Trends.Earth v2.2.6 — *Land degradation and SDG 15.3.1* ([docs](https://docs.trends.earth/en/v2.2.6/for_users/features/unccdreporting.html)), derived from UNCCD *Good Practice Guidance for SDG Indicator 15.3.1 v2 (2021)*, the *GPG Addendum (2025)*, and *GPG for national reporting on UNCCD Strategic Objective 3 (2021)*.

**Study area:** Morocco — 12 administrative regions (*découpage régional*), supplied as GeoJSON.

### What this notebook computes

| Block | Indicator | Output |
| :--- | :--- | :--- |
| **A** | SDG 15.3.1 — Land productivity (Trajectory / State / Performance) | 5-class + 3-class raster |
| **B** | SDG 15.3.1 — Land cover change | 3-class raster + transition matrix |
| **C** | SDG 15.3.1 — Soil organic carbon | 3-class raster + % SOC change |
| **D** | SDG 15.3.1 — 1OAO integration + Status matrix | Final indicator + Status |
| **F** | SO 3-1 — Drought hazard (SPI-12) | Class raster + tables |
| **G** | SO 3-3 — Drought vulnerability (DVI) | National/regional composite |

---

## Read this before running

Trajectory, State, Performance, and SOC are **multi-year statistics by construction**. Reporting values are anchored to fixed baselines (2000–2015) and reporting periods on the UNCCD 4-year cycle.

This notebook produces **reporting-compliant layers only (Blocks A–G)**: annual/period statistics, calendar-anchored, and suitable for UNCCD PRAIS submission.

## 0. Data source audit — Trends.Earth defaults vs. Google Earth Engine reality

Several Trends.Earth default inputs are **not available in the GEE public catalog**. Substitutes are coded below and flagged at runtime by the audit cell.

| Trends.Earth default | GEE availability | Substitute used here | Methodological cost |
| :--- | :--- | :--- | :--- |
| **MOD13Q1 coll. 6.1 NDVI 250 m** |  `MODIS/061/MOD13Q1` | — (selected engine) | None. *Terra is in orbital drift; plan VIIRS `NASA/VIIRS/002/VNP13A1` continuity for post-2026.* |
| **AVHRR/GIMMS NDVI 8 km** |  `NASA/GIMMS/3GV0` ends 2013 | Not used (MODIS selected) | Baselines before 2000 unavailable at 250 m |
| **ESA CCI Land Cover 300 m** |  Not in public catalog | `MODIS/061/MCD12Q1` (IGBP, 500 m, 2001–) for reporting; `GOOGLE/DYNAMICWORLD/V1` (10 m, ~5-day) for NRT | Different legend → crosswalk required; MCD12Q1 has known instability in sparse drylands |
| **SoilGrids 250 m SOC stock** |  Not official (ISRIC community assets) | `OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02` + bulk density → 0–30 cm stock | Different model/epoch; both are static, so the *relative* change logic is unaffected |
| **SoilGrids USDA soil units** (Performance strata) | | `OpenLandMap/SOL/SOL_GRTGROUP_USDA-SOILTAX_C/v01` | Coarser taxonomic split |
| **GPCC monitoring product** (SPI) |  Not in GEE | `UCSB-CHG/CHIRPS/DAILY` (5 km, 1981–, ~3-week latency; prelim ~2-day) | Better resolution; Morocco fully within 50°N/S so no coverage loss |
| **GPCP v2.3 / PERSIANN-CDR** |  PERSIANN in GEE, GPCP absent | CHIRPS primary, `ECMWF/ERA5_LAND/MONTHLY_AGGR` fallback | — |
| **MERRA-2 / ERA-I soil moisture** |  `NASA/GSFC/MERRA/lnd/2` | ERA5-Land volumetric soil water | — |
| **MOD16A2 ET (WUE)** |  `MODIS/061/MOD16A2GF` | — | Gap-filled product lags ~1 year |
| **JRC Drought Vulnerability Index** |  Not in GEE | Rebuilt DVI (Block G) from World Bank / national inputs, or manual asset upload | Not identical to JRC DVI — must be documented in the national report |

**HCP-specific recommendation:** for land cover, the MCD12Q1 crosswalk is a stopgap. A national land-cover series built from Sentinel-2 (10 m) with RGPH-consistent nomenclature would materially improve both the land-cover and SOC sub-indicators, and is the single highest-return upgrade for Morocco's next PRAIS cycle.

## 1. Environment

In [ ]:
%pip install -q earthengine-api geopandas pandas numpy scipy matplotlib requests pillow

In [ ]:
import json, math, os
import datetime as dt
from pathlib import Path
import ee
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print("ready |", dt.datetime.now().isoformat(timespec="seconds"))

ready | 2026-08-30T15:33:31


### 1.1 Central configurationEverything tunable lives here. Nothing below this cell should need editing for a routine run.

In [ ]:
CFG = {
    "GEE_PROJECT_ID": "degradation-finalee",
    "GEE_ASSET_ROOT": "projects/degradation-finalee/assets/ldn",

    "AOI_GEOJSON": "morocco_regions_12.geojson",
    "AOI_NAME_FIELD": "region",
    "AOI_CODE_FIELD": "cartodb_id",
    "AOI_FALLBACK": "FAO/GAUL_SIMPLIFIED_500m/2015/level1",
    "AOI_FALLBACK_FILTER": ("ADM0_NAME", "Morocco"),
    "AOI_SUBSET": None,

    "BASELINE_START": 2001,
    "BASELINE_END":   2015,
    "REPORT_START":   2016,
    "REPORT_END":     2025,
    "STATE_COMPARISON_YEARS": 3,

    "SCALE":1000,
    "SCALE_STATS": 1000,
    "SCALE_POP": 1000,
    "MAX_PIXELS": 1e13,
    "TILE_SCALE": 8,

    "NDVI_QA_LEVEL": 1,
    "TRAJ_METHOD": "restrend",
    "TRAJ_PVALUE": 0.05,
    "STATE_CLASS_METHOD": "equal_interval_extended",
    "PERF_RATIO_THRESHOLD": 0.5,

    "SOC_SOURCE": "openlandmap",
    "SOC_SOILGRIDS_ASSET": "projects/soilgrids-isric/ocs_mean",
    "SOC_CLIMATE_F": 0.80,
    "SOC_CHANGE_THRESHOLD": 0.10,
    "SOC_TRANSITION_YEARS": 20,

    "LC_SOURCE": "mcd12q1",
    "LC_CUSTOM_ASSET": None,

    "SPI_SOURCE": "chirps",
    "SPI_SCALE_MONTHS": 12,
    "SPI_REF_START": 1991,
    "SPI_REF_END": 2020,
    "SPI_METHOD": "empirical",
    "SPI_PIXEL_SCALE": 5566,

    "PRODUCT_TIER": "REPORTING",

    "EXPORT_CRS": "EPSG:4326",
    "EXPORT_PREFIX": "MAR_LDN",
}

BASELINE_YEARS = list(range(CFG["BASELINE_START"], CFG["BASELINE_END"] + 1))
REPORT_YEARS   = list(range(CFG["REPORT_START"],   CFG["REPORT_END" ] + 1))

print("baseline:", BASELINE_YEARS[0], "-", BASELINE_YEARS[-1],
      "| reporting:", REPORT_YEARS[0], "-", REPORT_YEARS[-1])

baseline: 2001 - 2015 | reporting: 2016 - 2025


### 1.2 Authentication`ee.Initialize` requires a Cloud project ID since November 2024. For an unattended / on-premisesscheduler, use a **service account** instead of the interactive flow (second block below) — that isthe pattern to use if you wire this into Celery.

In [ ]:
try:
    ee.Initialize(project=CFG["GEE_PROJECT_ID"])
except Exception:
    ee.Authenticate()
    ee.Initialize(project=CFG["GEE_PROJECT_ID"])

print("Earth Engine API:", ee.__version__)
print("project:", CFG["GEE_PROJECT_ID"])
print("round-trip test:", ee.Number(1).add(1).getInfo())

Earth Engine API: 1.7.39
project: degradation-finalee
round-trip test: 2


In [ ]:
'''
--- Unattended / server-side alternative (Celery, cron, Airflow) -------------
SERVICE_ACCOUNT = "ldn-runner@ee-hcp-ldn.iam.gserviceaccount.com"
KEY_FILE = "/etc/gee/ldn-runner.json"
credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, KEY_FILE)
ee.Initialize(credentials, project=CFG["GEE_PROJECT_ID"])
'''

## 2. Study extent — Morocco, 12 regions

**Expected GeoJSON schema** (FeatureCollection, EPSG:4326 / CRS84):

```json
{
  "type": "Feature",
  "properties": {
    "region_code": "04",
    "region_name_fr": "Rabat-Salé-Kénitra",
    "region_name_ar": "الرباط سلا القنيطرة"
  },
  "geometry": {...}
}
```

Sources, in order of preference for an HCP deliverable:

1. **HCP official boundaries** (RGPH 2024 cartographic base) — authoritative, aligns with all your other statistical products. Reproject to EPSG:4326 before upload.
2. **geoBoundaries gbOpen MAR ADM1** — current 12 regions, CC-BY. Note the files are served through Git-LFS, so a plain `raw.githubusercontent.com` fetch returns an LFS pointer, not the geometry; use [geoboundaries.org](https://www.geoboundaries.org/api/current/gbOpen/MAR/ADM1/) or the release ZIP.
3. **FAO GAUL / Natural Earth** — both still carry the **pre-2015 16-region** division. Usable as a geometric fallback only; never for a published regional breakdown.

Large geometries slow GEE down considerably. If your file is a high-fidelity coastline, simplify to ~250 m tolerance for statistics (`geom.simplify(250)`), keeping the full-resolution version for cartography.

In [ ]:
OFFICIAL_REGIONS_2015 = [
    ("01", "Tanger-Tétouan-Al Hoceïma",  "طنجة تطوان الحسيمة"),
    ("02", "L'Oriental",                 "الشرق"),
    ("03", "Fès-Meknès",                 "فاس مكناس"),
    ("04", "Rabat-Salé-Kénitra",         "الرباط سلا القنيطرة"),
    ("05", "Béni Mellal-Khénifra",       "بني ملال خنيفرة"),
    ("06", "Casablanca-Settat",          "الدار البيضاء سطات"),
    ("07", "Marrakech-Safi",             "مراكش آسفي"),
    ("08", "Drâa-Tafilalet",             "درعة تافيلالت"),
    ("09", "Souss-Massa",                "سوس ماسة"),
    ("10", "Guelmim-Oued Noun",          "كلميم واد نون"),
    ("11", "Laâyoune-Sakia El Hamra",    "العيون الساقية الحمراء"),
    ("12", "Dakhla-Oued Ed-Dahab",       "الداخلة وادي الذهب"),
]

REF_REGIONS = pd.DataFrame(OFFICIAL_REGIONS_2015,
                          columns=["region_code", "region_name_fr", "region_name_ar"])


REF_REGIONS

,region_code,region_name_fr,region_name_ar
0,01,Tanger-Tétouan-Al Hoceïma,طنجة تطوان الحسيمة
1,02,L'Oriental,الشرق
2,03,Fès-Meknès,فاس مكناس
3,04,Rabat-Salé-Kénitra,الرباط سلا القنيطرة
4,05,Béni Mellal-Khénifra,بني ملال خنيفرة
5,06,Casablanca-Settat,الدار البيضاء سطات
6,07,Marrakech-Safi,مراكش آسفي
7,08,Drâa-Tafilalet,درعة تافيلالت
8,09,Souss-Massa,سوس ماسة
9,10,Guelmim-Oued Noun,كلميم واد نون


In [ ]:
def load_aoi(cfg=CFG):
    
    path = Path(cfg["AOI_GEOJSON"])
    name_field = cfg["AOI_NAME_FIELD"]

    if path.exists():
        with open(path, encoding="utf-8") as fh:
            gj = json.load(fh)
        feats = gj["features"] if gj.get("type") == "FeatureCollection" else [gj]
        ee_feats = []
        for f in feats:
            props = dict(f.get("properties") or {})
            props = {k: ("" if v is None else v) for k, v in props.items()}
            ee_feats.append(ee.Feature(ee.Geometry(f["geometry"]), props))
        fc = ee.FeatureCollection(ee_feats)
        source = "GeoJSON: " + str(path)
    else:
        print("!! " + str(path) + " not found - falling back to " + cfg["AOI_FALLBACK"])
        print("!! WARNING: FAO GAUL carries the PRE-2015 16-region division.")
        fld, val = cfg["AOI_FALLBACK_FILTER"]
        fc = ee.FeatureCollection(cfg["AOI_FALLBACK"]).filter(ee.Filter.eq(fld, val))
        name_field = "ADM1_NAME"
        cfg["AOI_NAME_FIELD"] = name_field
        source = "FALLBACK " + cfg["AOI_FALLBACK"]

    if cfg["AOI_SUBSET"]:
        fc = fc.filter(ee.Filter.inList(name_field, cfg["AOI_SUBSET"]))

    names = fc.aggregate_array(name_field).getInfo()
    geom = fc.geometry().dissolve(maxError=100)

    print("source     :", source)
    print("features   :", len(names))
    print("regions    :", names)

    if len(names) != 12 and not cfg["AOI_SUBSET"]:
        print("!! expected 12 regions - check the decoupage vintage of your file")

    return fc, geom, names


REGIONS, AOI, REGION_NAMES = load_aoi()
AOI_AREA_KM2 = AOI.area(maxError=100).divide(1e6).getInfo()
print("total area : {:,.0f} km2".format(AOI_AREA_KM2))

source     : GeoJSON: morocco_regions_12.geojson
features   : 12
regions    : ['Laayoune-Saguia Hamra', 'Rabat-Sale-Kenitra', 'Beni Mellal-Khenifra', 'Dakhla-Oued Eddahab', 'Tanger-Tetouan-Hoceima', 'Marrakech-Safi', 'Daraa-Tafilelt', 'Guelmim-Oued Noun', 'Fes-Meknes', 'Oriental', 'Casablanca-Settat', 'Souss Massa']
total area : 680,630 km2


In [ ]:
import json

json.dumps(AOI.getInfo())

'{"type": "Polygon", "coordinates": [[[-8.668126, 26.787388], [-8.668126, 28.238474], [-8.668126, 28.597236000000002], [-8.703778, 28.701527], [-8.59088, 28.769259999999992], [-8.466098000000002, 28.831744], [-8.394795, 28.842154], [-8.400737, 28.956598999999997], [-8.347259, 28.925399000000002], [-8.305665000000001, 28.982591], [-8.222477, 29.008577], [-8.062043, 29.081303], [-7.996680999999998, 29.133218999999993], [-7.955087000000001, 29.179920999999997], [-7.860015, 29.231787], [-7.776826999999999, 29.288809], [-7.693639, 29.361337000000006], [-7.616393000000002, 29.397582000000003], [-7.509438000000001, 29.407935], [-7.438134000000001, 29.40793499999999], [-7.343062000000001, 29.449337], [-7.236106000000001, 29.537261000000004], [-7.1945120000000005, 29.568274999999996], [-7.170744000000001, 29.630273000000006], [-7.129150000000001, 29.645767], [-7.093498000000001, 29.676748000000003], [-6.9627740000000005, 29.614777], [-6.778572, 29.583778000000002], [-6.606255, 29.573443], [-6.5

In [ ]:
import json


with open("morocco_regions_12.geojson", encoding="utf-8") as f:
    data = json.load(f)

first_feature_props = data['features'][0]['properties']
print("Champs disponibles dans votre fichier :", list(first_feature_props.keys()))
print("Exemple de données :", first_feature_props)


possible_name_fields = [k for k in first_feature_props.keys() if 'name' in k.lower() or 'reg' in k.lower()]
print(f"\nSuggestion pour AOI_NAME_FIELD : {possible_name_fields}")

Champs disponibles dans votre fichier : ['cartodb_id', 'region', 'longitude', 'latitude']
Exemple de données : {'cartodb_id': 1, 'region': 'Laayoune-Saguia Hamra', 'longitude': -12.099233, 'latitude': 26.267329}

Suggestion pour AOI_NAME_FIELD : ['region']


### 2.1 Dataset availability audit
Run this before anything else. It confirms, against the live catalog, which of the Trends.Earth inputs actually resolve — so a missing dataset surfaces here rather than 40 cells later.

In [ ]:
CATALOG = {
    "NDVI MOD13Q1 (productivity)":        ("MODIS/061/MOD13Q1",                                  "ImageCollection", "core"),
    "NDVI VNP13A1 (post-MODIS backup)":   ("NASA/VIIRS/002/VNP13A1",                             "ImageCollection", "backup"),
    "NDVI GIMMS 3g (pre-2000 baseline)":  ("NASA/GIMMS/3GV0",                                    "ImageCollection", "optional"),
    "Land cover MCD12Q1":                 ("MODIS/061/MCD12Q1",                                  "ImageCollection", "core"),
    "Land cover Dynamic World (NRT)":     ("GOOGLE/DYNAMICWORLD/V1",                             "ImageCollection", "core"),
    "Land cover ESA WorldCover":          ("ESA/WorldCover/v200",                                "ImageCollection", "optional"),
    "SOC OpenLandMap":                    ("OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02", "Image",           "core"),
    "Bulk density OpenLandMap":           ("OpenLandMap/SOL/SOL_BULKDENS-FINEEARTH_USDA-4A1H_M/v02", "Image",       "core"),
    "Soil taxonomy OpenLandMap":          ("OpenLandMap/SOL/SOL_GRTGROUP_USDA-SOILTAX_C/v01",    "Image",           "core"),
    "SOC SoilGrids (community)":          (CFG["SOC_SOILGRIDS_ASSET"],                           "Image",           "optional"),
    "Precipitation CHIRPS daily":         ("UCSB-CHG/CHIRPS/DAILY",                              "ImageCollection", "core"),
    "Precipitation PERSIANN-CDR":         ("NOAA/PERSIANN-CDR",                                  "ImageCollection", "optional"),
    "Precipitation ERA5-Land monthly":    ("ECMWF/ERA5_LAND/MONTHLY_AGGR",                       "ImageCollection", "core"),
    "Evapotranspiration MOD16A2GF":       ("MODIS/061/MOD16A2GF",                                "ImageCollection", "optional"),
    "Population WorldPop age/sex":        ("WorldPop/GP/100m/pop_age_sex_cons_unadj",            "ImageCollection", "core"),
    "Population WorldPop total":          ("WorldPop/GP/100m/pop",                               "ImageCollection", "optional"),
    "Population GHSL":                    ("JRC/GHSL/P2023A/GHS_POP",                            "ImageCollection", "optional"),
    "Water JRC surface water":            ("JRC/GSW1_4/GlobalSurfaceWater",                      "Image",           "optional"),
    "Aridity index (SOC climate zone)":   ("projects/sat-io/open-datasets/global_ai_et0",        "ImageCollection", "optional"),
}

def audit_catalog(catalog=CATALOG):
    rows = []
    for label, (asset, kind, role) in catalog.items():
        status, detail = "OK", ""
        try:
            info = ee.data.getAsset(asset)
            detail = info.get("type", "")
        except Exception as exc:
            status = "MISSING"
            detail = str(exc).split("\n")[0][:70]
        rows.append({"dataset": label, "asset_id": asset, "expected": kind,
                     "role": role, "status": status, "detail": detail})
    df = pd.DataFrame(rows)
    blockers = df[(df.status == "MISSING") & (df.role == "core")]
    if len(blockers):
        print("!! CORE DATASETS UNAVAILABLE - substitute before proceeding:")
        print(blockers[["dataset", "asset_id"]].to_string(index=False))
    else:
        print("all core datasets resolved")
    return df

AUDIT = audit_catalog()
AUDIT[["dataset", "role", "status"]]

all core datasets resolved


,dataset,role,status
0,NDVI MOD13Q1 (productivity),core,OK
1,NDVI VNP13A1 (post-MODIS backup),backup,OK
2,NDVI GIMMS 3g (pre-2000 baseline),optional,OK
3,Land cover MCD12Q1,core,OK
4,Land cover Dynamic World (NRT),core,OK
5,Land cover ESA WorldCover,optional,OK
6,SOC OpenLandMap,core,OK
7,Bulk density OpenLandMap,core,OK
8,Soil taxonomy OpenLandMap,core,OK
9,SOC SoilGrids (community),optional,OK


# BLOCK A — Land productivity
Three metrics computed from the annual NDVI integral, then combined into a 5-class and a 3-class layer (the 3-class layer is what SDG 15.3.1 requires).

Trends.Earth defines the "annual integral of NDVI" as the **mean annual NDVI**, for interpretability. We keep that definition to stay comparable with Trends.Earth outputs.

## A.1 NDVI Engine (MOD13Q1 Collection 6.1)

**QA handling:** `SummaryQA`

* `0` = good
* `1` = marginal
* `2` = snow/ice
* `3` = cloudy

We keep values **≤ 1**, which is the Trends.Earth convention.

In the pre-Saharan and Saharan regions, NDVI hovers near the sensor noise floor (**0.05–0.12**). Trends there are weakly identifiable and should be interpreted using the **bare-soil mask in Block B**, rather than on their own.


In [ ]:
MOD13Q1 = ee.ImageCollection("MODIS/061/MOD13Q1")
VNP13A1 = ee.ImageCollection("NASA/VIIRS/002/VNP13A1")

def _mask_mod13(img):
    qa = img.select("SummaryQA")
    keep = qa.lte(CFG["NDVI_QA_LEVEL"])
    ndvi = img.select("NDVI").multiply(0.0001).updateMask(keep).rename("ndvi")
    return ndvi.copyProperties(img, ["system:time_start"])

def ndvi_collection(start, end):
    
    return MOD13Q1.filterDate(start, end).filterBounds(AOI).map(_mask_mod13)

def annual_ndvi(year):
    
    coll = ndvi_collection(str(year) + "-01-01", str(year + 1) + "-01-01")
    img = coll.mean().rename("ndvi")
    return (img.set("year", year)
               .set("n_obs", coll.size())
               .set("system:time_start", ee.Date.fromYMD(year, 7, 1).millis()))

def annual_ndvi_collection(years):
    return ee.ImageCollection([annual_ndvi(y) for y in years])

NDVI_BASELINE = annual_ndvi_collection(BASELINE_YEARS)
NDVI_REPORT   = annual_ndvi_collection(REPORT_YEARS)
NDVI_ALL      = annual_ndvi_collection(BASELINE_YEARS + REPORT_YEARS)


print("composites per year (first/last baseline year):",
      annual_ndvi(BASELINE_YEARS[0]).get("n_obs").getInfo(),
      annual_ndvi(REPORT_YEARS[-1]).get("n_obs").getInfo())

composites per year (first/last baseline year): 23 23


## A.2 Climate-correction inputs (precipitation, evapotranspiration) Required for RESTREND / RUE / WUE.
**For Morocco this is not optional in practice** — the 1980–2020 precipitation decline is strong enough that an uncorrected NDVI trajectory will attribute climate-driven productivity loss to land management, and inflate the degraded proportion.Run the uncorrected trajectory *and*  RESTREND, and report the difference.

In [ ]:
CHIRPS = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
MOD16  = ee.ImageCollection("MODIS/061/MOD16A2GF")

def annual_precip(year):
    img = (CHIRPS.filterDate(str(year) + "-01-01", str(year + 1) + "-01-01")
           .sum().rename("precip"))
    return img.set("year", year).set("system:time_start",
                                     ee.Date.fromYMD(year, 7, 1).millis())

def annual_et(year):
    coll = MOD16.filterDate(str(year) + "-01-01", str(year + 1) + "-01-01").select("ET")
    
    coll = coll.map(lambda i: i.updateMask(i.lt(32760)).multiply(0.1))
    return (coll.sum().rename("et").set("year", year)
            .set("system:time_start", ee.Date.fromYMD(year, 7, 1).millis()))

PRECIP_ALL = ee.ImageCollection([annual_precip(y) for y in BASELINE_YEARS + REPORT_YEARS])

## A.3 Productivity **Trajectory**
Pixel-level linear regression over the analysis period + Mann-Kendall non-parametric significance test. Significant (p ≤ 0.05) positive slope → improving (+1); significant negative → degrading (−1); otherwise stable (0).

Four variants, selected by `CFG["TRAJ_METHOD"]`:

| Method       | Regressed variable              | When to use                                                    |
| :----------- | :------------------------------ | :------------------------------------------------------------- |
| `ndvi_trend` | annual NDVI integral            | default, no climate correction                                 |
| `restrend`   | residual of NDVI ~ precipitation| **recommended for Morocco**; requires a stable NDVI–P relationship |
| `rue`        | NDVI / precipitation            | simple, assumes linear water-use response                      |
| `wue`        | NDVI / evapotranspiration       | drier systems where P is a poor proxy for available water      |

RESTREND caveat (Wessels et al. 2012): where degradation is *gradual and pervasive*, the residual regression can absorb the degradation signal itself and under-detect. Do not use RESTREND alone as evidence of "no degradation".

In [ ]:
def _mk_bands(mk_img):
    
    names = mk_img.bandNames()
    tau_n = names.filter(ee.Filter.stringContains("item", "tau")).get(0)
    p_n   = names.filter(ee.Filter.stringContains("item", "p-value")).get(0)
    return mk_img.select([tau_n], ["tau"]), mk_img.select([p_n], ["p_value"])

def trend_and_significance(coll, band="ndvi"):
    
    withx = coll.map(lambda i: ee.Image.constant(ee.Number(i.get("year")))
                     .float().rename("year")
                     .addBands(i.select(band).rename("value"))
                     .copyProperties(i, ["year", "system:time_start"]))

    lf = withx.select(["year", "value"]).reduce(ee.Reducer.linearFit())
    slope = lf.select("scale").rename("slope")
    sen = withx.select(["year", "value"]).reduce(ee.Reducer.sensSlope()).select("slope").rename("sen_slope")

    mk = coll.select(band).reduce(ee.Reducer.kendallsCorrelation(1))
    tau, pval = _mk_bands(mk)

    return slope.addBands(sen).addBands(tau).addBands(pval)

def classify_trajectory(stats, p_thresh=None):
    p_thresh = p_thresh or CFG["TRAJ_PVALUE"]
    sig = stats.select("p_value").lte(p_thresh)
    up = stats.select("slope").gt(0).And(sig)
    dn = stats.select("slope").lt(0).And(sig)
    return (ee.Image(0).where(up, 1).where(dn, -1)
            .updateMask(stats.select("slope").mask())
            .rename("traj").toInt8())

def build_trajectory(years, method=None):
    method = method or CFG["TRAJ_METHOD"]
    ndvi = annual_ndvi_collection(years)

    if method == "ndvi_trend":
        target = ndvi
    elif method == "rue":
        def _rue(y):
            r = annual_ndvi(y).divide(annual_precip(y).max(1)).rename("ndvi")
            return r.set("year", y)
        target = ee.ImageCollection([_rue(y) for y in years])
    elif method == "wue":
        def _wue(y):
            r = annual_ndvi(y).divide(annual_et(y).max(0.1)).rename("ndvi")
            return r.set("year", y)
        target = ee.ImageCollection([_wue(y) for y in years])
    elif method == "restrend":
        
        pairs = ee.ImageCollection([
            annual_precip(y).rename("precip").addBands(annual_ndvi(y).rename("ndvi")).set("year", y)
            for y in years])
        fit = pairs.select(["precip", "ndvi"]).reduce(ee.Reducer.linearFit())
        a, b = fit.select("scale"), fit.select("offset")

        
        def _resid(y):
            pred = annual_precip(y).multiply(a).add(b)
            r = annual_ndvi(y).subtract(pred).rename("ndvi")
            return r.set("year", y)
        target = ee.ImageCollection([_resid(y) for y in years])
    else:
        raise ValueError("unknown TRAJ_METHOD: " + str(method))

    stats = trend_and_significance(target)
    return classify_trajectory(stats), stats

TRAJ_REPORT, TRAJ_STATS = build_trajectory(REPORT_YEARS)
print("trajectory method:", CFG["TRAJ_METHOD"], "| years:", REPORT_YEARS[0], "-", REPORT_YEARS[-1])

trajectory method: restrend | years: 2016 - 2025


## A.4 Productivity **State**

Detects short-term change against the baseline distribution:

1. Baseline annual NDVI distribution per pixel → 10 classes;
2. Baseline mean NDVI → class (1–10);
3. Comparison period (last 3 years) mean NDVI → class (1–10);
4. Difference ≤ −2 → degrading, ≥ +2 → improving, otherwise stable.

> **Documentation note.** The Trends.Earth page states "if the difference … is ≤ 2 … potentially degraded. If the difference is ≥ 2 … recent improvement". Taken literally that is contradictory; the intended rule (and the implemented one) is **≤ −2 degraded / ≥ +2 improved**.

> **Implementation note.** The page describes extending the baseline distribution by 5% at both extremes and using the "cut-off values of the 10 percentile classes" — which is ambiguous between equal-interval bins on the extended range and empirical deciles. Both are implemented; switch with `CFG["STATE_CLASS_METHOD"]`. Default is `equal_interval_extended`, which follows the 5%-extension text. Expect a few percent of pixels to differ between the two — document your choice in the metadata of the national report.

In [ ]:
def _class_from_breaks(value_img, breaks):
    
    cls = ee.Image(1)
    for b in breaks:
        cls = cls.add(value_img.gt(b))
    return cls.rename("class").toInt8()

def build_state(baseline_years=None, n_compare=None):
    baseline_years = baseline_years or BASELINE_YEARS
    n_compare = n_compare or CFG["STATE_COMPARISON_YEARS"]
    compare_years = REPORT_YEARS[-n_compare:]

    bl = annual_ndvi_collection(baseline_years)

    if CFG["STATE_CLASS_METHOD"] == "percentile":
        pcts = list(range(10, 100, 10))
        p_img = bl.reduce(ee.Reducer.percentile(pcts))
        breaks = [p_img.select(i) for i in range(len(pcts))]
    else:
        mn, mx = bl.min(), bl.max()
        rng = mx.subtract(mn)
        lo = mn.subtract(rng.multiply(0.05))     
        hi = mx.add(rng.multiply(0.05))
        step = hi.subtract(lo).divide(10)
        breaks = [lo.add(step.multiply(k)) for k in range(1, 10)]

    bl_mean = bl.mean()
    cp_mean = annual_ndvi_collection(compare_years).mean()

    bl_cls = _class_from_breaks(bl_mean, breaks)
    cp_cls = _class_from_breaks(cp_mean, breaks)

    diff = cp_cls.subtract(bl_cls).rename("state_diff")
    state = (ee.Image(0).where(diff.lte(-2), -1).where(diff.gte(2), 1)
             .updateMask(bl_mean.mask()).rename("state").toInt8())

    return state, diff, bl_cls, cp_cls, compare_years

STATE, STATE_DIFF, STATE_BL_CLS, STATE_CP_CLS, COMPARE_YEARS = build_state()
print("state comparison period:", COMPARE_YEARS)

state comparison period: [2023, 2024, 2025]


## A.5 Productivity **Performance**

**Local productivity relative to the best-performing land of the same *ecological potential*.**

Similar-unit strata = land cover × soil taxonomic unit.

The 90th percentile of mean NDVI within each stratum is taken as the attainable maximum; pixels below **50%** of it are considered degraded.

> **Substitution:**  
> Trends.Earth uses **SoilGrids USDA soil units (250 m)** and **ESA CCI land cover (300 m)**. Neither is available in the GEE public catalog. We use **OpenLandMap USDA great groups (250 m)** and the **MCD12Q1-derived UNCCD 7-class layer**. Strata are therefore coarser, which makes the 90th percentile *less* locally specific and tends to be conservative in heterogeneous terrain—relevant for the Atlas ranges, less so for the plains.
>
> **Scaling:**  
> The grouped reduction is the heaviest computation in the notebook. Over the full national extent at **250 m**, it can time out. Options, in order:
> 1. Raise `TILE_SCALE` to **16**.
> 2. Run `PERF_SCALE = 500`.
> 3. Loop region by region and mosaic.
>
> A region-wise loop is provided.

In [ ]:
SOILTAX = ee.Image("OpenLandMap/SOL/SOL_GRTGROUP_USDA-SOILTAX_C/v01").select(0).rename("soil")
PERF_SCALE = 500     

def build_performance(years=None, lc_image=None, geometry=None, scale=None):
    years = years or REPORT_YEARS
    geometry = geometry if geometry is not None else AOI
    scale = scale or PERF_SCALE

    ndvi_mean = annual_ndvi_collection(years).mean().rename("ndvi")
    lc = lc_image if lc_image is not None else LC_UNCCD_BASELINE   

    units = lc.multiply(1000).add(SOILTAX).rename("unit").toInt()
    units = units.updateMask(ndvi_mean.mask()).updateMask(lc.mask())

    grouped = ee.Dictionary(
        ndvi_mean.addBands(units).reduceRegion(
            reducer=ee.Reducer.percentile([90]).setOutputs(["p90"])
                      .group(groupField=1, groupName="unit"),
            geometry=geometry, scale=scale,
            maxPixels=CFG["MAX_PIXELS"], tileScale=CFG["TILE_SCALE"], bestEffort=True)
    ).get("groups")

    glist = ee.List(grouped)
    keys = glist.map(lambda d: ee.Number(ee.Dictionary(d).get("unit")).toInt())
    vals = glist.map(lambda d: ee.Number(ee.Dictionary(d).get("p90")))

    unit_max = units.remap(keys, vals, -9999).rename("unit_max")
    unit_max = unit_max.updateMask(unit_max.neq(-9999))

    ratio = ndvi_mean.divide(unit_max).rename("perf_ratio")
    perf = (ee.Image(0).where(ratio.lt(CFG["PERF_RATIO_THRESHOLD"]), -1)
            .updateMask(ratio.mask()).rename("perf").toInt8())

    return perf, ratio, unit_max



## A.6 Combining the Three Productivity Metrics

Trends.Earth uses an **18-row lookup table** (Trajectory × State × Performance) to produce a **5-class** productivity layer, which is then collapsed into the **3-class** layer required for reporting.

### 5-class codes

- `1` — Declining
- `2` — Moderate decline
- `3` — Stressed
- `4` — Stable
- `5` — Improving

### 3-class codes

- `-1` — Degrading
- `0` — Stable
- `+1` — Improving

In [ ]:

PROD_LOOKUP_5 = {
    ( 1,  1,  0): 5, ( 1,  1, -1): 5,
    ( 1,  0,  0): 5, ( 1,  0, -1): 5,
    ( 1, -1,  0): 5, ( 1, -1, -1): 2,
    ( 0,  1,  0): 4, ( 0,  1, -1): 4,
    ( 0,  0,  0): 4, ( 0,  0, -1): 3,
    ( 0, -1,  0): 2, ( 0, -1, -1): 1,
    (-1,  1,  0): 1, (-1,  1, -1): 1,
    (-1,  0,  0): 1, (-1,  0, -1): 1,
    (-1, -1,  0): 1, (-1, -1, -1): 1,
}

FIVE_TO_THREE = {1: -1, 2: -1, 3: 0, 4: 0, 5: 1}

def combine_productivity(traj, state, perf):
    
    codes = (traj.add(1).multiply(100)
             .add(state.add(1).multiply(10))
             .add(perf.add(1))).toInt()

    frm = [(t + 1) * 100 + (s + 1) * 10 + (p + 1) for (t, s, p) in PROD_LOOKUP_5]
    to5 = [PROD_LOOKUP_5[k] for k in PROD_LOOKUP_5]

    prod5 = codes.remap(frm, to5, 0).rename("prod_5class").toInt8()
    prod3 = prod5.remap(list(FIVE_TO_THREE), list(FIVE_TO_THREE.values()), 0) \
                 .updateMask(prod5.neq(0)).rename("productivity").toInt8()

    return prod3, prod5

# BLOCK B — Land Cover

The land cover indicator is computed in **three steps**:

1. Reclassify the input land cover to the **UNCCD 7-class** legend.
2. Compute the land cover transition between two dates.
3. Score each transition as **degradation**, **stable**, or **improvement** using an editable transition matrix.

### UNCCD 7-Class Legend

| Code | Class |
|------:|-------|
| `1` | Tree-covered |
| `2` | Grassland |
| `3` | Cropland |
| `4` | Wetland |
| `5` | Artificial |
| `6` | Other land (bare) |
| `7` | Water |

> **Substitution:**  
> ESA CCI Land Cover is not available in the GEE public catalog. We therefore use **MCD12Q1 (IGBP, 500 m, 2001–)**.
>
> This has two important consequences for Morocco:
>
> 1. **Sparse drylands:** MCD12Q1 is unstable from year to year in sparse-vegetation drylands, creating spurious **grassland ↔ bare land** transitions across the pre-Saharan belt. A temporal mode filter is applied below to reduce this effect.
> 2. **Shrublands:** The shrubland classes have no direct equivalent in the UNCCD legend and are therefore mapped to **grassland**, following the ESA CCI convention. This choice is nevertheless debatable for *steppe à alfa*.
>
> Both assumptions are exposed below as editable constants.

In [ ]:
MCD12Q1 = ee.ImageCollection("MODIS/061/MCD12Q1")
DW = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")


IGBP_TO_UNCCD = {
    1: 1, 2: 1, 3: 1, 4: 1, 5: 1,   
    6: 2, 7: 2,                     
    8: 1, 9: 2,                     
    10: 2,                          
    11: 4,                          
    12: 3,                          
    13: 5,                          
    14: 3,                          
    15: 6,                          
    16: 6,                          
    17: 7                           
}

UNCCD_LABELS = {
    1: "Tree-covered", 2: "Grassland", 3: "Cropland", 4: "Wetland",
    5: "Artificial", 6: "Other land", 7: "Water"
}

MCD12Q1_FIRST, MCD12Q1_LAST = 2001, 2023

def landcover_unccd(year, smooth_years=3):
    
    y = max(MCD12Q1_FIRST, min(int(year), MCD12Q1_LAST))
    half = smooth_years // 2
    lo, hi = max(MCD12Q1_FIRST, y - half), min(MCD12Q1_LAST, y + half)

    coll = (MCD12Q1.filterDate(str(lo) + "-01-01", str(hi + 1) + "-01-01")
            .select("LC_Type1")
            .map(lambda i: i.remap(list(IGBP_TO_UNCCD), list(IGBP_TO_UNCCD.values()), 0)))

    lc = coll.mode() if smooth_years > 1 else coll.first()
    lc = lc.updateMask(lc.gt(0)).rename("lc").toInt8()
    return lc.set("year", y)

def landcover_dynamicworld(start, end):
    
    DW_TO_UNCCD = {0: 7, 1: 1, 2: 2, 3: 4, 4: 3, 5: 2, 6: 5, 7: 6, 8: 6}
    lab = DW.filterDate(start, end).filterBounds(AOI).select("label").mode()
    return (lab.remap(list(DW_TO_UNCCD), list(DW_TO_UNCCD.values()), 0)
            .selfMask().rename("lc").toInt8())

LC_UNCCD_BASELINE = landcover_unccd(CFG["BASELINE_START"])
LC_UNCCD_BLEND    = landcover_unccd(CFG["BASELINE_END"])
LC_UNCCD_TARGET   = landcover_unccd(CFG["REPORT_END"])

print("land cover years:", CFG["BASELINE_START"], CFG["BASELINE_END"], CFG["REPORT_END"],
      "(clamped to MCD12Q1", MCD12Q1_FIRST, "-", MCD12Q1_LAST, ")")

land cover years: 2001 2015 2025 (clamped to MCD12Q1 2001 - 2023 )


### B.1 Transition Matrix — **Requires National Expert Validation**

GPG **Step 3** is explicit: the degradation meaning of each transition must reflect **local land degradation processes**.

The matrix below uses the **UNCCD/Trends.Earth default** transition scores. It should be considered a **starting point**, not a national position.

- **Rows:** initial land cover class
- **Columns:** final land cover class
- **Scores:**
  - `-1` — Degradation
  - `0` — No change
  - `+1` — Improvement

### Transitions Worth Reviewing for Morocco

- **Grassland → Cropland (`−1` by default)**  
  Cereal expansion onto rangeland in the **Saïss** and **Chaouia** represents a productivity gain but also a soil carbon loss. The default therefore classifies this transition as **degradation**.

- **Other land → Cropland (`+1` by default)**  
  Irrigated expansion onto bare land in **Souss-Massa** and **Drâa-Tafilalet** is scored as an **improvement**, although it may rely on non-renewable groundwater extraction. LDN accounting is land-based and does not capture this effect, so it should be acknowledged in the accompanying narrative.

- **Grassland ↔ Other land**  
  These transitions are largely an artifact of **MCD12Q1** classification noise in the pre-Saharan belt. The temporal mode filter applied above reduces, but does not completely eliminate, this issue.

In [ ]:

LC_TRANSITION_MATRIX = np.array([
    
    [    0,   -1,   -1,  -1,  -1,   -1,    0],   
    [    1,    0,   -1,  -1,  -1,   -1,    0],   
    [    1,    1,    0,   0,  -1,   -1,    0],   
    [    1,    1,   -1,    0,  -1,   -1,    0],   
    [    1,    1,    1,    1,   0,    0,    0],   
    [    1,    1,    1,    1,  -1,    0,    0],   
    [    0,    0,    0,    0,  -1,    0,    0],   
], dtype=int)

LC_MATRIX_DF = pd.DataFrame(LC_TRANSITION_MATRIX,
                            index=[UNCCD_LABELS[i] for i in range(1, 8)],
                            columns=[UNCCD_LABELS[i] for i in range(1, 8)])
LC_MATRIX_DF

,Tree-covered,Grassland,Cropland,Wetland,Artificial,Other land,Water
Tree-covered,0,-1,-1,-1,-1,-1,0
Grassland,1,0,-1,-1,-1,-1,0
Cropland,1,1,0,0,-1,-1,0
Wetland,1,1,-1,0,-1,-1,0
Artificial,1,1,1,1,0,0,0
Other land,1,1,1,1,-1,0,0
Water,0,0,0,0,-1,0,0


In [ ]:
def build_landcover_indicator(lc_start, lc_end, matrix=LC_TRANSITION_MATRIX):
    
    trans = lc_start.multiply(10).add(lc_end).rename("transition").toInt()

    frm, to = [], []
    for i in range(7):
        for j in range(7):
            frm.append((i + 1) * 10 + (j + 1))
            to.append(int(matrix[i, j]))

    lc_ind = (trans.remap(frm, to, 0)
              .updateMask(lc_start.mask().And(lc_end.mask()))
              .rename("landcover").toInt8())

    return lc_ind, trans

LC_IND_REPORT, LC_TRANS_REPORT = build_landcover_indicator(LC_UNCCD_BLEND, LC_UNCCD_TARGET)
LC_IND_BASELINE, LC_TRANS_BASE = build_landcover_indicator(LC_UNCCD_BASELINE, LC_UNCCD_BLEND)
WATER_MASK = LC_UNCCD_TARGET.eq(7)

print("land cover sub-indicator built")

land cover sub-indicator built


# BLOCK C — Soil Organic Carbon

This block implements the **combined land-cover / SOC method**.

**SoilGrids 0–30 cm SOC stock** is the Trends.Earth reference. Here, an equivalent stock is reconstructed from **OpenLandMap organic carbon content** and **bulk density**, integrated over the **0–30 cm** soil profile.

> **Structural limitation — state this in your report.**
>
> Because no SOC time series exists, the relative SOC change is **entirely determined** by the land-cover transition and the elapsed time:
>
> ```text
> ΔSOC/SOC = (coef − 1) × min(n, 20) / 20
> ```
>
> The SOC sub-indicator is therefore **not independent evidence** of degradation. It is a re-expression of **Block B** through the application of IPCC coefficients.
>
> Under **1OAO**, it can only add degraded area where land cover has already identified a transition. Trends.Earth has the same property; this is a characteristic of the **method**, not of this implementation.

> **Practical consequence**
>
> For a **9-year reporting period** (`n/20 = 0.45`), only transitions with a coefficient **≤ 0.78** or **≥ 1.22** can exceed the **±10%** threshold.
>
> As a result, **f-coefficient transitions** (e.g., cropland conversions with **f = 0.80** in **Temperate Dry**) fall **just below** the threshold and are therefore classified as **stable**. This is an artifact of the reporting-period length and is worth noting in a footnote.

In [ ]:
def soc_stock_0_30():
    if CFG["SOC_SOURCE"] == "soilgrids_asset":
        
        return ee.Image(CFG["SOC_SOILGRIDS_ASSET"]).select(0).rename("soc")

    oc = ee.Image("OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02")   
    bd = ee.Image("OpenLandMap/SOL/SOL_BULKDENS-FINEEARTH_USDA-4A1H_M/v02")  

    def layer(depth_a, depth_b, thickness_m):
        oc_m = oc.select(depth_a).add(oc.select(depth_b)).divide(2).divide(5.0)     
        bd_m = bd.select(depth_a).add(bd.select(depth_b)).divide(2).multiply(10.0)  
        
        return oc_m.multiply(bd_m).multiply(thickness_m).divide(100.0)

    soc = layer("b0", "b10", 0.10).add(layer("b10", "b30", 0.20))
    return soc.rename("soc").toFloat()

SOC_REF = soc_stock_0_30()
print("SOC reference stock (t/ha) built from:", CFG["SOC_SOURCE"])

SOC reference stock (t/ha) built from: openlandmap


### C.1 IPCC / UNCCD Land-Use Conversion Coefficients

The coefficient **`f`** is the **IPCC climate-zone factor** applied to land-use transitions involving cropland.

| Climate zone | `f` |
|---------------|----:|
| Temperate Dry | **0.80** |
| Temperate Moist | 0.69 |
| Tropical Dry | 0.58 |
| Tropical Moist | 0.48 |
| Tropical Montane | 0.64 |

> **Morocco:**  
> **Temperate Dry (`f = 0.80`)** is appropriate for the overwhelming majority of Morocco under the IPCC climate classification. This includes Mediterranean and arid/semi-arid regions, with mean annual temperatures below **18 °C** in the north and warm-temperate dry conditions in the south.
>
> If a spatially varying **`f`** is preferred, provide a climate-zone raster and use:
>
> ```python
> soc_coefficient_image(f_image=...)
> ```
>
> By default, a single constant coefficient is applied across the study area, which is also the approach used by **Trends.Earth** for most national assessments.

In [ ]:
def soc_coefficient_matrix(f=None):
    f = f if f is not None else CFG["SOC_CLIMATE_F"]
    m = np.array([
        
        [      1.0,  1.0,   f,       1.0,    0.1,  0.1,   1.0],   
        [      1.0,  1.0,   f,       1.0,    0.1,  0.1,   1.0],   
        [    1.0/f, 1.0/f, 1.0,  1.0/0.71,   0.1,  0.1,   1.0],   
        [      1.0,  1.0,  0.71,     1.0,    0.1,  0.1,   1.0],   
        [      2.0,  2.0,  2.0,      2.0,    1.0,  1.0,   1.0],   
        [      2.0,  2.0,  2.0,      2.0,    1.0,  1.0,   1.0],   
        [      1.0,  1.0,  1.0,      1.0,    1.0,  1.0,   1.0],   
    ], dtype=float)
    return m

def build_soc_indicator(lc_start, lc_end, n_years, soc_ref=None):
    soc_ref = soc_ref if soc_ref is not None else SOC_REF
    m = soc_coefficient_matrix()
    trans = lc_start.multiply(10).add(lc_end).toInt()

    frm = [(i + 1) * 10 + (j + 1) for i in range(7) for j in range(7)]
    to = [float(m[i, j]) for i in range(7) for j in range(7)]

    coef = trans.remap(frm, to, 1.0).rename("coef")
    frac = min(n_years, CFG["SOC_TRANSITION_YEARS"]) / float(CFG["SOC_TRANSITION_YEARS"])

    rel_change = coef.subtract(1).multiply(frac).rename("soc_rel_change")
    soc_end = soc_ref.multiply(rel_change.add(1)).rename("soc_end")

    thr = CFG["SOC_CHANGE_THRESHOLD"]
    soc_ind = (ee.Image(0)
               .where(rel_change.lte(-thr), -1)
               .where(rel_change.gte(thr), 1)
               .updateMask(soc_ref.mask().And(lc_start.mask()))
               .rename("soc").toInt8())

    return soc_ind, rel_change, soc_end

N_YEARS_REPORT = CFG["REPORT_END"] - CFG["REPORT_START"] + 1
N_YEARS_BASE   = CFG["BASELINE_END"] - CFG["BASELINE_START"] + 1

SOC_IND_REPORT, SOC_REL_REPORT, SOC_END = build_soc_indicator(LC_UNCCD_BLEND, LC_UNCCD_TARGET, N_YEARS_REPORT)
SOC_IND_BASELINE, SOC_REL_BASE, _ = build_soc_indicator(LC_UNCCD_BASELINE, LC_UNCCD_BLEND, N_YEARS_BASE)

print("SOC change window:", N_YEARS_REPORT, "years -> attenuation factor", round(min(N_YEARS_REPORT, 20) / 20.0, 3))

SOC change window: 10 years -> attenuation factor 0.5


# BLOCK A (Completed) — Performance + Productivity Integration

The **Performance** sub-indicator requires the land-cover strata produced in **Block B**. For that reason, it is computed here, completing the productivity workflow.

In [ ]:

PERF, PERF_RATIO, PERF_UNITMAX = build_performance(REPORT_YEARS, lc_image=LC_UNCCD_BLEND)
PRODUCTIVITY, PRODUCTIVITY_5 = combine_productivity(TRAJ_REPORT, STATE, PERF)
print("Productivity indicators (PRODUCTIVITY) defined.")

Productivity indicators (PRODUCTIVITY) defined.


# BLOCK D — SDG 15.3.1 Integration (1OAO) and Status

### One-Out-All-Out (1OAO)

Under the **One-Out-All-Out (1OAO)** rule:

- A pixel is classified as **degraded** if **any** sub-indicator is degraded.
- A pixel is classified as **improving** only if **at least one** sub-indicator is improving **and none** are degrading.

### Status

The **Status** map combines the **baseline-period** indicator with the **reporting-period** indicator using:

- the **3 × 3** transition matrix defined in the **GPG Addendum**, and
- the expanded **9-class** version, which distinguishes **persistent** from **recent** changes.

In [ ]:
def one_out_all_out(prod, lc, soc):
    stack = ee.Image.cat([prod.rename("p"), lc.rename("l"), soc.rename("s")])
    mn = stack.reduce(ee.Reducer.min())
    mx = stack.reduce(ee.Reducer.max())
    sdg = (ee.Image(0).where(mx.eq(1), 1).where(mn.eq(-1), -1)
           .updateMask(stack.mask().reduce(ee.Reducer.min()))
           .rename("sdg1531").toInt8())
    return sdg


SDG_REPORT = one_out_all_out(PRODUCTIVITY, LC_IND_REPORT, SOC_IND_REPORT)
print("SDG 15.3.1 report indicator (SDG_REPORT) ready.")

SDG 15.3.1 report indicator (SDG_REPORT) ready.


### D.1 Baseline-Period Indicator

The **Status** matrix requires **SDG 15.3.1** to be computed **for the baseline period itself**, as specified in the GPG Addendum:

> *Baseline Assessment = Status 2015*

This requires re-running the **entire SDG 15.3.1 workflow** using the baseline years, making it the most computationally expensive step in the notebook.

> **Recommendation:**  
> Export the baseline indicator once as a **Google Earth Engine asset** and reload it in subsequent reporting cycles instead of recomputing it each time.

In [ ]:
RECOMPUTE_BASELINE = True
BASELINE_ASSET = CFG["GEE_ASSET_ROOT"] + "/sdg1531_baseline_2001_2015"

def one_out_all_out(prod, lc, soc):
    stack = ee.Image.cat([prod.rename("p"), lc.rename("l"), soc.rename("s")])
    mn = stack.reduce(ee.Reducer.min())
    mx = stack.reduce(ee.Reducer.max())
    sdg = (ee.Image(0).where(mx.eq(1), 1).where(mn.eq(-1), -1)
           .updateMask(stack.mask().reduce(ee.Reducer.min()))
           .rename("sdg1531").toInt8())
    return sdg

if RECOMPUTE_BASELINE:
    traj_bl, _ = build_trajectory(BASELINE_YEARS)
    bl_ref_years = BASELINE_YEARS[:-CFG["STATE_COMPARISON_YEARS"]]
    bl_cmp_years = BASELINE_YEARS[-CFG["STATE_COMPARISON_YEARS"]:]
    bl_ref = annual_ndvi_collection(bl_ref_years)
    mn, mx = bl_ref.min(), bl_ref.max()
    rng = mx.subtract(mn)
    lo, hi = mn.subtract(rng.multiply(0.05)), mx.add(rng.multiply(0.05))
    step = hi.subtract(lo).divide(10)
    breaks_bl = [lo.add(step.multiply(k)) for k in range(1, 10)]
    st_bl = _class_from_breaks(bl_ref.mean(), breaks_bl)
    st_cp = _class_from_breaks(annual_ndvi_collection(bl_cmp_years).mean(), breaks_bl)
    d = st_cp.subtract(st_bl)
    state_bl = ee.Image(0).where(d.lte(-2), -1).where(d.gte(2), 1).rename("state").toInt8()
    perf_bl, _, _ = build_performance(BASELINE_YEARS, lc_image=LC_UNCCD_BASELINE)
    prod_bl, prod5_bl = combine_productivity(traj_bl, state_bl, perf_bl)
    SDG_BASELINE = one_out_all_out(prod_bl, LC_IND_BASELINE, SOC_IND_BASELINE)
else:
    SDG_BASELINE = ee.Image(BASELINE_ASSET).rename("sdg1531").toInt8()

print("Baseline indicator (SDG_BASELINE) ready.")

Baseline indicator (SDG_BASELINE) ready.


In [ ]:





STATUS_3 = {
    (-1, -1): -1,
    (-1,  0): -1,
    (-1,  1):  1,
    ( 0, -1): -1,
    ( 0,  0):  0,
    ( 0,  1):  1,
    ( 1, -1): -1,
    ( 1,  0):  1,
    ( 1,  1):  1,
}



STATUS_EXPANDED = {
    (-1, -1): 1,  
    (-1,  0): 3,  
    (-1,  1): 6,  
    ( 0, -1): 2,  
    ( 0,  0): 4,  
    ( 0,  1): 6,  
    ( 1, -1): 2,  
    ( 1,  0): 5,  
    ( 1,  1): 7,  
}


STATUS_EXPANDED_LABELS = {
    1: "Persistent degradation",
    2: "Recent degradation",
    3: "Baseline degradation",
    4: "Stability",
    5: "Baseline improvement",
    6: "Recent improvement",
    7: "Persistent improvement",
}


def build_status(sdg_baseline, sdg_period):
    """
    Build SDG 15.3.1 Status layers from baseline and reporting-period indicators.
    """

    keys = [
        (b, p)
        for b in (-1, 0, 1)
        for p in (-1, 0, 1)
    ]  

    
    code = (
        sdg_baseline.add(1)
        .multiply(10)
        .add(sdg_period.add(1))
        .toInt()
    )

    frm = [
        (b + 1) * 10 + (p + 1)
        for (b, p) in keys
    ]

    status3 = (
        code.remap(
            frm,
            [STATUS_3[k] for k in keys],
            0
        )
        .rename("status")
        .toInt8()
    )

    status9 = (
        code.remap(
            frm,
            [STATUS_EXPANDED[k] for k in keys],
            0
        )
        .rename("status_expanded")
        .toInt8()
    )

    return status3, status9



STATUS, STATUS_EXP = build_status(
    SDG_BASELINE,
    SDG_REPORT
)

print("Status maps ready")

Status maps ready


## D.2 Area statistics by region`pixelArea()` on an equal-area-safe reduction. Note that GEE computes pixel area on the ellipsoid,so no projection choice is needed for area accounting; `EXPORT_CRS` only affects the exported raster.

In [ ]:
CLASS_LABELS_3 = {-1: "Degraded", 0: "Stable", 1: "Improved"}

def area_by_class(class_img, regions=None, labels=None, scale=None, name="indicator"):
    regions = regions if regions is not None else REGIONS
    scale = scale or CFG["SCALE_STATS"]
    labels = labels or CLASS_LABELS_3
    field = CFG["AOI_NAME_FIELD"]

    img = ee.Image.pixelArea().divide(1e4).rename("ha").addBands(class_img.rename("class"))

    fc = img.reduceRegions(
        collection=regions,
        reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
        scale=scale,
        tileScale=CFG["TILE_SCALE"]
    )

    rows = []
    for feat in fc.getInfo()["features"]:
        props = feat["properties"]
        region = props.get(field, "?")
        for g in props.get("groups", []):
            rows.append({
                "region": region,
                "class_code": int(g["class"]),
                "class": labels.get(int(g["class"]), str(g["class"])),
                "area_ha": g["sum"]
            })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    tot = df.groupby("region")["area_ha"].transform("sum")
    df["pct"] = 100 * df["area_ha"] / tot
    df["indicator"] = name
    return df.sort_values(["region", "class_code"]).reset_index(drop=True)

def pivot_pct(df):
    return (df.pivot_table(index="region", columns="class", values="pct", fill_value=0)
              .round(2))

SDG_STATS = area_by_class(SDG_REPORT, name="SDG 15.3.1 " + str(CFG["REPORT_START"]) + "-" + str(CFG["REPORT_END"]))
display(pivot_pct(SDG_STATS))

class,Degraded,Improved,Stable
region,,,
Beni Mellal-Khenifra,76.53,0.97,22.50
Casablanca-Settat,89.93,0.21,9.87
Dakhla-Oued Eddahab,21.22,0.00,78.78
Daraa-Tafilelt,14.68,0.57,84.75
Fes-Meknes,66.69,1.54,31.76
Guelmim-Oued Noun,63.93,0.01,36.06
Laayoune-Saguia Hamra,26.15,0.09,73.76
Marrakech-Safi,87.29,1.09,11.62
Oriental,70.64,0.10,29.26


In [ ]:


FRAMES = []

for img, name in [
    (PRODUCTIVITY, "Productivity"),
    (LC_IND_REPORT, "Land cover"),
    (SOC_IND_REPORT, "Soil organic carbon"),
    (SDG_REPORT, "SDG 15.3.1"),
    (SDG_BASELINE, "SDG 15.3.1 baseline"),
]:
    FRAMES.append(
        area_by_class(
            img,
            name=name
        )
    )



SUBIND_STATS = pd.concat(
    FRAMES,
    ignore_index=True
)



STATUS_STATS = area_by_class(
    STATUS,
    name="Status"
)

STATUS_EXP_STATS = area_by_class(
    STATUS_EXP,
    labels=STATUS_EXPANDED_LABELS,
    name="Status expanded"
)



national = (
    SUBIND_STATS
    .groupby(["indicator", "class"])["area_ha"]
    .sum()
    .reset_index()
)

national["pct"] = (
    national["area_ha"]
    / national.groupby("indicator")["area_ha"].transform("sum")
    * 100
)



national.pivot_table(
    index="indicator",
    columns="class",
    values="pct",
    fill_value=0
).round(2)

class,Degraded,Improved,Stable
indicator,,,
Land cover,3.20,2.61,94.19
Productivity,44.06,0.00,55.94
SDG 15.3.1,45.32,0.39,54.29
SDG 15.3.1 baseline,13.14,3.95,82.91
Soil organic carbon,2.98,2.58,94.44


--->'problème dans la source des données'

--->'problème dans la source des données'

# BLOCK F — SO 3.1 (Drought Hazard)

This block implements the **GPG SO3 procedure**:

1. Compute **SPI-12**.
2. Convert drought intensity into drought classes.
3. Calculate the proportion of land affected by each drought class for each year.

> **Substitution:**  
> The Trends.Earth default uses **SPI derived from the GPCC Monitoring Product**, which is not available in the GEE public catalog.
>
> We use **CHIRPS** (**5 km, 1981–present**) instead. Trends.Earth also supports CHIRPS, and it is a suitable choice for Morocco because:
>
> - Morocco lies entirely within the **CHIRPS 50°N–50°S coverage range**.
> - The **5 km spatial resolution** captures the Atlas orographic gradient better than a **1° gauge-based product**.
>
> **ERA5-Land** is included as a fallback precipitation dataset.

---

> **Numerical deviation — document this choice.**
>
> The GPG specifies:
>
> 1. A **gamma distribution fit**.
> 2. Transformation using the **Abramowitz & Stegun normal approximation**.
>
> However, Earth Engine does not provide the required **incomplete gamma function**, meaning that the exact gamma CDF cannot be evaluated server-side.
>
> Two alternative implementations are therefore provided:

### 1. `empirical` (default)

A **non-parametric SPI** approach:

- The observed 12-month precipitation total is ranked within the **1991–2020 reference distribution** for the same calendar month.
- The **Gringorten plotting position** is applied.
- Values are transformed using the **Abramowitz & Stegun inverse normal approximation**.

This is a recognised SPI variant and avoids gamma-fit instability in arid environments where the distribution is poorly represented.

### 2. `gamma_wh`

A closer approximation to the GPG method:

- Uses a **method-of-moments gamma fit**.
- Applies the **Wilson–Hilferty cube-root approximation** to estimate the gamma CDF.

This approach is closer in spirit to the GPG specification but is most reliable for **shape parameter α > 1** and becomes less accurate in very dry pixels.

> **Expected differences:**  
> Differences of approximately **0.05–0.15 SPI units** may occur between the two approaches in the sub-humid north, with larger differences possible in Saharan areas.
>
> Whichever implementation is selected, document it in the **PRAIS metadata**. Reviewers may compare the national SPI results against the UNCCD default dataset, so the methodological difference should be explicitly justified.

In [ ]:



def monthly_precip_collection(y0, y1):
    """
    Build monthly precipitation collection for the selected SPI source.
    """

    if CFG["SPI_SOURCE"] == "era5land":
        src = (
            ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
            .select("total_precipitation_sum")
        )

        return (
            src
            .filterDate(
                str(y0) + "-01-01",
                str(y1 + 1) + "-01-01"
            )
            .map(
                lambda i: (
                    i.multiply(1000)
                    .rename("precip")
                    .copyProperties(
                        i,
                        ["system:time_start"]
                    )
                )
            )
        )

    
    months = []

    for y in range(y0, y1 + 1):
        for m in range(1, 13):

            start = ee.Date.fromYMD(y, m, 1)

            img = (
                CHIRPS
                .filterDate(
                    start,
                    start.advance(1, "month")
                )
                .sum()
                .rename("precip")
                .set(
                    "system:time_start",
                    start.millis()
                )
                .set("year", y)
                .set("month", m)
            )

            months.append(img)

    return ee.ImageCollection(months)



def accum_sum(end_date, n_months=None):
    """
    Compute precipitation accumulation over n months
    ending at end_date (exclusive).
    """

    n = n_months or CFG["SPI_SCALE_MONTHS"]

    end = ee.Date(end_date)
    start = end.advance(-n, "month")

    if CFG["SPI_SOURCE"] == "era5land":

        src = (
            ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
            .select("total_precipitation_sum")
            .filterDate(start, end)
        )

        return (
            src
            .sum()
            .multiply(1000)
            .rename("acc")
        )

    return (
        CHIRPS
        .filterDate(start, end)
        .sum()
        .rename("acc")
    )



def inv_norm_cdf(p):
    """
    Abramowitz & Stegun 26.2.23 rational approximation.
    Absolute error < 4.5e-4.
    """

    c0, c1, c2 = 2.515517, 0.802853, 0.010328
    d1, d2, d3 = 1.432788, 0.189269, 0.001308

    p = p.clamp(1e-6, 1 - 1e-6)

    lower = p.lte(0.5)

    pp = p.where(
        lower.Not(),
        ee.Image(1).subtract(p)
    )

    
    t = (
        pp.pow(2)
        .pow(-1)
        .log()
        .sqrt()
    )

    num = (
        t.multiply(c1)
        .add(t.pow(2).multiply(c2))
        .add(c0)
    )

    den = (
        t.multiply(d1)
        .add(t.pow(2).multiply(d2))
        .add(t.pow(3).multiply(d3))
        .add(1)
    )

    z = t.subtract(
        num.divide(den)
    )

    return (
        z.multiply(-1)
        .where(lower.Not(), z)
        .rename("spi")
    )

In [ ]:
def accum_sum(end_date, n_months=None):
    n = n_months or CFG["SPI_SCALE_MONTHS"]
    end = ee.Date(end_date)
    start = end.advance(-n, "month")
    if CFG["SPI_SOURCE"] == "era5land":
        src = (ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
               .select("total_precipitation_sum").filterDate(start, end))
        return src.sum().multiply(1000).rename("acc")
    return CHIRPS.filterDate(start, end).sum().rename("acc")

def inv_norm_cdf(p):
    c0, c1, c2 = 2.515517, 0.802853, 0.010328
    d1, d2, d3 = 1.432788, 0.189269, 0.001308
    p = p.clamp(1e-6, 1 - 1e-6)
    lower = p.lte(0.5)
    pp = p.where(lower.Not(), ee.Image(1).subtract(p))
    t = pp.pow(2).pow(-1).log().sqrt()
    num = t.multiply(c1).add(t.pow(2).multiply(c2)).add(c0)
    den = t.multiply(d1).add(t.pow(2).multiply(d2)).add(t.pow(3).multiply(d3)).add(1)
    z = t.subtract(num.divide(den))
    return z.multiply(-1).where(lower.Not(), z).rename("spi")

def spi(end_date, n_months=None, method=None):
    n = n_months or CFG["SPI_SCALE_MONTHS"]
    method = method or CFG["SPI_METHOD"]
    end = ee.Date(end_date)
    month = end.get("month").getInfo()
    day = end.get("day").getInfo()
    obs = accum_sum(end, n)
    ref_years = range(CFG["SPI_REF_START"], CFG["SPI_REF_END"] + 1)
    refs = [accum_sum(ee.Date.fromYMD(y, month, day), n) for y in ref_years]
    ref_coll = ee.ImageCollection(refs)
    n_ref = len(refs)
    if method == "gamma_wh":
        mean = ref_coll.mean()
        var = ref_coll.reduce(ee.Reducer.variance())
        alpha = mean.pow(2).divide(var.max(1e-6))
        beta = var.divide(mean.max(1e-6))
        ratio = obs.divide(alpha.multiply(beta).max(1e-6)).max(1e-9)
        z = (ratio.pow(1.0 / 3.0)
             .subtract(ee.Image(1).subtract(alpha.multiply(9).pow(-1)))
             .multiply(alpha.multiply(9).sqrt())).rename("spi")
        return z.updateMask(alpha.gt(0.1))
    count = ref_coll.map(lambda i: i.lte(obs)).sum()
    p = count.subtract(0.44).divide(n_ref + 0.12)
    return inv_norm_cdf(p)

DROUGHT_CLASSES = {0: "No drought", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Extreme"}

def drought_class(spi_img):
    return (ee.Image(0)
            .where(spi_img.lte(0).And(spi_img.gt(-1.0)), 1)
            .where(spi_img.lte(-1.0).And(spi_img.gt(-1.5)), 2)
            .where(spi_img.lte(-1.5).And(spi_img.gt(-2.0)), 3)
            .where(spi_img.lte(-2.0), 4)
            .updateMask(spi_img.mask()).rename("drought").toInt8())

SPI_2025 = spi(str(CFG["REPORT_END"] + 1) + "-01-01")
DROUGHT_2025 = drought_class(SPI_2025)
print("SPI and drought classification ready.")

SPI and drought classification ready.


### F.1 SO 3.1 — Proportion of Land per Drought Class (Annual Series)

For each reporting year, the drought indicator is derived from the **December SPI-12 value**, as required by the GPG.

The December SPI-12 integrates precipitation over the full preceding **12-month Gregorian calendar period**, providing the annual drought condition used for reporting.

In [ ]:
def so3_1_series(years, regions=None, scale=None):
    regions = regions if regions is not None else REGIONS
    scale = scale or CFG["SPI_PIXEL_SCALE"]
    frames = []

    for y in years:
        dc = drought_class(spi(str(y + 1) + "-01-01"))
        df = area_by_class(dc, regions=regions, labels=DROUGHT_CLASSES,
                           scale=scale, name="SO3-1 " + str(y))
        if df.empty:
            continue
        df["year"] = y
        frames.append(df)
        print("  computed", y)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()




SO3_1_YEARS = REPORT_YEARS
SO3_1 = so3_1_series(SO3_1_YEARS)

if not SO3_1.empty:
    display(SO3_1.pivot_table(index="year", columns="class", values="area_ha",
                              aggfunc="sum").div(1e6).round(3))

  computed 2016
  computed 2017
  computed 2018
  computed 2019
  computed 2020
  computed 2021
  computed 2022
  computed 2023
  computed 2024
  computed 2025


class,Extreme,Mild,Moderate,No drought,Severe
year,,,,,
2016,NaN,11.496,1.372,54.511,0.016
2017,8.696,19.184,20.473,11.287,7.754
2018,0.003,3.442,0.075,63.875,NaN
2019,5.532,37.996,15.637,4.130,4.100
2020,6.499,27.828,17.395,9.711,5.961
2021,4.134,38.127,12.349,9.429,3.356
2022,1.021,31.201,6.796,27.275,1.102
2023,15.936,20.109,12.254,14.327,4.768
2024,9.818,24.485,9.573,19.554,3.964


In [ ]:
import matplotlib.pyplot as plt


def check_precipitation_totals(region_name):
    region_fc = REGIONS.filter(ee.Filter.eq(CFG['AOI_NAME_FIELD'], region_name))

    
    ref_years = list(range(1991, 2021))
    ref_precip = ee.ImageCollection([annual_precip(y) for y in ref_years]).mean()

    
    precip_2024 = annual_precip(2024)

    stats = ee.Image.cat([ref_precip.rename('avg'), precip_2024.rename('y2024')]) \
        .reduceRegion(reducer=ee.Reducer.mean(), geometry=region_fc.geometry(), scale=5000).getInfo()

    print(f"--- Diagnostic pour {region_name} ---")
    print(f"Moyenne historique (CHIRPS): {stats['avg']:.2f} mm")
    print(f"Total année 2024 (CHIRPS): {stats['y2024']:.2f} mm")
    print(f"Rapport: {100 * stats['y2024']/stats['avg']:.1f}%")


try:
    check_precipitation_totals(REGION_NAMES[5]) 
except Exception as e:
    print(f"Erreur diagnostic: {e}")

--- Diagnostic pour Marrakech-Safi ---
Moyenne historique (CHIRPS): 305.94 mm
Total année 2024 (CHIRPS): 210.43 mm
Rapport: 68.8%


# BLOCK G — SO 3.3: Drought Vulnerability Index

> **This dataset does not exist in Earth Engine.**
>
> The JRC DVI (Carrão et al. 2016) is not published as a GEE asset, and it is a
> *national-level* composite for most of its 15 inputs — only gROADS and FAO
> irrigated land are gridded.
>
> Three viable paths are available:
>
> 1. **Upload the JRC layer** as a GEE asset if you obtain it from the JRC Global
>    Drought Observatory, and point `DVI_ASSET` at it. This provides the highest
>    comparability with other Parties' reports.
>
> 2. **Rebuild the composite** from the original World Bank / FAO / OECD indicators
>    (implemented below). This approach is reproducible and transparent, but it is
>    *your* index, not the JRC's — state this explicitly.
>
> 3. **Build a regional DVI from HCP sources** — the methodologically strongest
>    option for Morocco and the one I would defend in a national report.
>
>    The Carrão framework requires orthogonal social, economic, and infrastructural
>    factors. RGPH 2024 and the ENCDM provide literacy, rural share, dependency
>    ratio, water access, and multidimensional poverty *at commune level*, which is
>    two orders of magnitude finer than anything the global product offers.
>    A template is provided below.
>
> **Aggregation** follows Carrão:
>
> - Data Envelopment Analysis (DEA) is applied per factor using
>   **benefit-of-the-doubt weighting**.
> - The three factor scores are then combined using their arithmetic mean.
>
> Therefore:
>
> `DVI = (Soc + Econ + Infr) / 3`

In [ ]:


DVI_ASSET = None




DVI_INDICATORS = {
    
    
    

    "agriculture_pct_gdp": (
        "NV.AGR.TOTL.ZS",
        "economic",
        +1,
    ),
    "gdp_per_capita": (
        "NY.GDP.PCAP.CD",
        "economic",
        -1,
    ),
    "poverty_headcount": (
        "SI.POV.DDAY",
        "economic",
        +1,
    ),
    "rural_population_pct": (
        "SP.RUR.TOTL.ZS",
        "social",
        +1,
    ),
    "literacy_rate": (
        "SE.ADT.LITR.ZS",
        "social",
        -1,
    ),
    "life_expectancy": (
        "SP.DYN.LE00.IN",
        "social",
        -1,
    ),
    "pop_15_64_pct": (
        "SP.POP.1564.TO.ZS",
        "social",
        -1,
    ),
    "water_access_rural": (
        "SH.H2O.BASW.RU.ZS",
        "social",
        -1,
    ),
    "gov_effectiveness": (
        "GE.EST",
        "social",
        -1,
    ),
    "irrigated_land_pct": (
        "AG.LND.IRIG.AG.ZS",
        "infrastructural",
        -1,
    ),
    "water_stress": (
        "ER.H2O.FWST.ZS",
        "infrastructural",
        +1,
    ),
}


def fetch_worldbank(
    codes,
    countries="MAR;DZA;TUN;EGY;MRT;ESP;PRT",
    start=2015,
    end=2023,
):
    """
    Retrieve World Bank indicators for the selected countries and years.

    Returns:
        A DataFrame indexed by ISO3 country code, with the latest
        available value for each indicator within the requested period.
    """

    import requests

    out = {}

    for name, (code, _, _) in codes.items():

        url = (
            "https://api.worldbank.org/v2/country/"
            + countries
            + "/indicator/"
            + code
            + "?format=json&per_page=2000&date="
            + str(start)
            + ":"
            + str(end)
        )

        try:
            response = requests.get(
                url,
                timeout=30,
            )

            js = response.json()

            recs = (
                js[1]
                if len(js) > 1 and js[1]
                else []
            )

            df = pd.DataFrame(
                [
                    {
                        "iso3": r["countryiso3code"],
                        "year": int(r["date"]),
                        name: r["value"],
                    }
                    for r in recs
                    if r["value"] is not None
                ]
            )

            if not df.empty:
                out[name] = (
                    df
                    .sort_values("year")
                    .groupby("iso3")
                    .last()[[name]]
                )

        except Exception as exc:
            print(
                "  ! "
                + name
                + ": "
                + str(exc)[:60]
            )

    return (
        pd.concat(out.values(), axis=1)
        if out
        else pd.DataFrame()
    )

In [ ]:



def bod_dea(X):
    """
    Benefit-of-the-Doubt DEA composite (output-oriented, unit input).

    Parameters
    ----------
    X : pandas.DataFrame
        Normalised indicators (0..1), with rows = units.

    Returns
    -------
    pandas.Series
        DEA scores normalised to 0..1.
    """

    try:
        from scipy.optimize import linprog
    except ImportError:
        print("  scipy unavailable - falling back to arithmetic mean")
        return X.mean(axis=1)

    Y = X.values

    n, m = Y.shape
    scores = []

    for k in range(n):
        
        res = linprog(
            c=-Y[k],
            A_ub=Y,
            b_ub=np.ones(n),
            bounds=[(0, None)] * m,
            method="highs",
        )

        scores.append(
            -res.fun
            if res.success
            else np.nan
        )

    s = pd.Series(
        scores,
        index=X.index
    )

    return s / s.max()



def build_dvi(
    df,
    spec=DVI_INDICATORS
):
    """
    Build the Direction-aware Vulnerability Index (DVI).

    Indicators are min-max normalised so that higher values
    consistently represent greater vulnerability.
    """

    norm = pd.DataFrame(
        index=df.index
    )

    
    for name, (_, factor, direction) in spec.items():

        if name not in df.columns:
            continue

        col = df[name].astype(float)

        if col.notna().sum() < 2:
            continue

        z = (
            (col - col.min())
            / (col.max() - col.min())
        )

        norm[name] = (
            z
            if direction > 0
            else 1 - z
        )

    
    factors = {}

    for factor in [
        "social",
        "economic",
        "infrastructural",
    ]:

        cols = [
            name
            for name, (_, f, _) in spec.items()
            if f == factor
            and name in norm.columns
        ]

        if cols:
            factors[factor] = bod_dea(
                norm[cols].fillna(
                    norm[cols].mean()
                )
            )

    F = pd.DataFrame(factors)

    
    F["DVI"] = F.mean(axis=1)

    return (
        F.round(3),
        norm
    )




try:
    WB = fetch_worldbank(
        DVI_INDICATORS
    )

    DVI_TABLE, DVI_NORM = build_dvi(
        WB
    )

    display(
        DVI_TABLE.sort_values(
            "DVI",
            ascending=False
        )
    )

    print(
        "\nDVI is a REBUILT composite, not the JRC index. "
        "Document the difference in PRAIS."
    )

except Exception as exc:
    print(
        "World Bank fetch failed (offline?):",
        str(exc)[:100]
    )

    DVI_TABLE = None

,social,economic,infrastructural,DVI
iso3,,,,
EGY,1.000,0.958,0.972,0.977
MRT,1.000,1.000,0.452,0.817
MAR,0.690,0.946,0.778,0.805
DZA,0.497,0.897,1.000,0.798
TUN,0.483,0.942,0.648,0.691
PRT,0.634,0.187,0.032,0.284
ESP,0.025,0.041,0.233,0.100



DVI is a REBUILT composite, not the JRC index. Document the difference in PRAIS.


In [ ]:

try:
    print("Fetching World Bank indicators...")
    WB_DATA = fetch_worldbank(DVI_INDICATORS)

    if not WB_DATA.empty:
        print("Computing DVI composite...")
        DVI_TABLE, DVI_NORM = build_dvi(WB_DATA)
        display(DVI_TABLE.sort_values("DVI", ascending=False))
        print("\nDVI calculated successfully. This is a rebuilt composite based on Carrão et al. (2016).")
    else:
        print("No data retrieved from World Bank API.")
except Exception as e:
    print(f"Error during DVI computation: {str(e)}")

Fetching World Bank indicators...
Computing DVI composite...


,social,economic,infrastructural,DVI
iso3,,,,
EGY,1.000,0.958,0.972,0.977
MRT,1.000,1.000,0.452,0.817
MAR,0.690,0.946,0.778,0.805
DZA,0.497,0.897,1.000,0.798
TUN,0.483,0.942,0.648,0.691
PRT,0.634,0.187,0.032,0.284
ESP,0.025,0.041,0.233,0.100



DVI calculated successfully. This is a rebuilt composite based on Carrão et al. (2016).


## D.3 Classification SDG 15.3.1 annuelle par région (2016-2025)

Le calcul ci-dessus (`SDG_REPORT`) ne produit qu'**une seule** classification pour toute la
période 2016-2025 comparée globalement à la baseline 2001-2015.

Conformément à la **Good Practice Guidance SDG 15.3.1 v2.0 (UNCCD, agence "custodian" de
l'indicateur)** et à l'implémentation de référence **Trends.Earth**, le statut de dégradation
doit être évalué **pour la baseline et pour chaque année de suivi**, avec une **règle de
persistance** (une terre reste classée "dégradée" tant qu'elle ne s'est pas améliorée par
rapport à la baseline / évaluation précédente).

La cellule suivante recalcule les **3 sous-indicateurs complets** (productivité, couverture des
terres, carbone organique du sol) pour **chaque année de 2016 à 2025**, puis les combine via la
règle **1OAO**, exactement comme pour `SDG_REPORT`, mais avec :

- **Trajectoire** : tendance NDVI cumulative de {REPORT_START} jusqu'à l'année évaluée (fenêtre
  qui s'agrandit chaque année, comme c'est l'usage pour un test de Mann-Kendall).
- **État** et **Performance** : fenêtre de comparaison glissante de
  `CFG["STATE_COMPARISON_YEARS"]` ans se terminant à l'année évaluée (tronquée au début de
  la période de reporting pour les toutes premières années).
- **Couverture des terres** et **COS** : baseline (2001) vs. couverture des terres de l'année
  évaluée.

**Coût de calcul** : ceci relance la réduction groupée la plus lourde du notebook
(`build_performance`, Bloc A.5) **10 fois** (une fois par année) plutôt qu'une seule — la
cellule peut prendre plusieurs minutes.


In [ ]:
def build_state_for(compare_years, baseline_years=None):
    """Same logic as build_state(), but with an explicit comparison window
    instead of always REPORT_YEARS[-n_compare:]."""
    baseline_years = baseline_years or BASELINE_YEARS
    bl = annual_ndvi_collection(baseline_years)

    if CFG["STATE_CLASS_METHOD"] == "percentile":
        pcts = list(range(10, 100, 10))
        p_img = bl.reduce(ee.Reducer.percentile(pcts))
        breaks = [p_img.select(i) for i in range(len(pcts))]
    else:
        mn, mx = bl.min(), bl.max()
        rng = mx.subtract(mn)
        lo = mn.subtract(rng.multiply(0.05))
        hi = mx.add(rng.multiply(0.05))
        step = hi.subtract(lo).divide(10)
        breaks = [lo.add(step.multiply(k)) for k in range(1, 10)]

    bl_mean = bl.mean()
    cp_mean = annual_ndvi_collection(compare_years).mean()

    bl_cls = _class_from_breaks(bl_mean, breaks)
    cp_cls = _class_from_breaks(cp_mean, breaks)

    diff = cp_cls.subtract(bl_cls).rename("state_diff")
    state = (ee.Image(0).where(diff.lte(-2), -1).where(diff.gte(2), 1)
             .updateMask(bl_mean.mask()).rename("state").toInt8())
    return state


def sdg1531_status_for_year(end_year):
    """Full 3-sub-indicator SDG 15.3.1 classification, as of a given year,
    following the same 1OAO combination as SDG_REPORT/one_out_all_out()."""
    traj_years = list(range(CFG["REPORT_START"], end_year + 1))
    traj, _ = build_trajectory(traj_years)

    n_compare = min(CFG["STATE_COMPARISON_YEARS"], len(traj_years))
    compare_years = traj_years[-n_compare:]

    state = build_state_for(compare_years)

    lc_end = landcover_unccd(end_year)
    perf, _, _ = build_performance(compare_years, lc_image=lc_end)
    prod3, _ = combine_productivity(traj, state, perf)

    lc_ind, _ = build_landcover_indicator(LC_UNCCD_BASELINE, lc_end)

    n_years_soc = end_year - CFG["BASELINE_END"]
    soc_ind, _, _ = build_soc_indicator(LC_UNCCD_BASELINE, lc_end, n_years_soc)

    return one_out_all_out(prod3, lc_ind, soc_ind)


def sdg1531_annual_series(years, regions=None, scale=None):
    regions = regions if regions is not None else REGIONS
    scale = scale or CFG["SCALE_STATS"]
    frames = []
    for y in years:
        sdg_y = sdg1531_status_for_year(y)
        df = area_by_class(sdg_y, regions=regions, scale=scale,
                           name="SDG 15.3.1 " + str(y))
        if df.empty:
            continue
        df["year"] = y
        frames.append(df)
        print("  computed", y)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


SDG_ANNUAL_STATS = sdg1531_annual_series(REPORT_YEARS)

if not SDG_ANNUAL_STATS.empty:
    display(SDG_ANNUAL_STATS.pivot_table(index="year", columns="class",
                                         values="area_ha", aggfunc="sum")
                             .div(1e6).round(3))


  computed 2017
  computed 2018
  computed 2019
  computed 2020
  computed 2021
  computed 2022
  computed 2023
  computed 2024
  computed 2025


class,Degraded,Improved,Stable
year,,,
2017,16.264,2.164,48.832
2018,10.694,2.289,54.283
2019,10.568,2.271,54.429
2020,15.233,1.960,50.075
2021,19.385,1.540,46.344
2022,27.640,0.928,38.701
2023,28.726,0.880,37.664
2024,32.075,0.775,34.420
2025,30.138,0.955,36.177


# BLOCK I — Exports, Visualisation, and Provenance

In [ ]:
def export_image(image, name, region=None, scale=None, to="asset"):
    region = region if region is not None else AOI
    scale = scale or CFG["SCALE"]
    desc = CFG["EXPORT_PREFIX"] + "_" + name

    img = image.clip(region).set({
        "product_tier": CFG["PRODUCT_TIER"] or "REPORTING",
        "methodology": "Trends.Earth v2.2.6 / UNCCD GPG 15.3.1 v2 (2021) + Addendum (2025)",
        "baseline_period": str(CFG["BASELINE_START"]) + "-" + str(CFG["BASELINE_END"]),
        "reporting_period": str(CFG["REPORT_START"]) + "-" + str(CFG["REPORT_END"]),
        "traj_method": CFG["TRAJ_METHOD"],
        "spi_method": CFG["SPI_METHOD"],
        "generated": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    })

    common = dict(image=img, description=desc, region=region, scale=scale,
                  crs=CFG["EXPORT_CRS"], maxPixels=int(CFG["MAX_PIXELS"]))

    if to == "asset":
        task = ee.batch.Export.image.toAsset(assetId=CFG["GEE_ASSET_ROOT"] + "/" + name, **common)
    elif to == "drive":
        task = ee.batch.Export.image.toDrive(folder="LDN_Morocco", fileNamePrefix=desc, **common)
    else:
        raise ValueError("to must be 'asset' or 'drive'")

    task.start()
    print("started:", desc, "->", to)
    return task

In [ ]:



OUT = Path("outputs")
OUT.mkdir(exist_ok=True)




SUBIND_STATS.to_csv(
    OUT / "sdg1531_subindicators_by_region.csv",
    index=False,
)

STATUS_STATS.to_csv(
    OUT / "status_by_region.csv",
    index=False,
)

STATUS_EXP_STATS.to_csv(
    OUT / "status_expanded_by_region.csv",
    index=False,
)




if not SO3_1.empty:
    SO3_1.to_csv(
        OUT / "so3_1_drought_land_proportion.csv",
        index=False,
    )


if not SDG_ANNUAL_STATS.empty:
    SDG_ANNUAL_STATS.to_csv(
        OUT / "sdg1531_degradation_by_region_annual.csv",
        index=False,
    )




if DVI_TABLE is not None:
    DVI_TABLE.to_csv(
        OUT / "so3_3_dvi_rebuilt.csv"
    )




PROVENANCE = {
    "indicator": (
        "SDG 15.3.1 / UNCCD SO2 / UNCCD SO3"
    ),

    "methodology": (
        "Trends.Earth v2.2.6; "
        "UNCCD GPG 15.3.1 v2 (2021); "
        "GPG Addendum (2025); "
        "GPG-SO3 (2021)"
    ),

    "producer": (
        "HCP - Haut-Commissariat au Plan"
    ),

    "gsbpm_phase": (
        "5.4 Impute/derive new variables; "
        "5.6 Calculate aggregates"
    ),

    "spatial_reference": CFG["EXPORT_CRS"],

    "extent": (
        "Morocco - 12 regions "
    ),

    "periods": {
        "baseline": [
            CFG["BASELINE_START"],
            CFG["BASELINE_END"],
        ],
        "reporting": [
            CFG["REPORT_START"],
            CFG["REPORT_END"],
        ],
    },

    "substitutions": {
        "land_cover": (
            "MODIS MCD12Q1 (IGBP) instead of ESA CCI "
            "- not available in GEE"
        ),

        "soil_organic_carbon": (
            "OpenLandMap instead of SoilGrids "
            "- not available in GEE"
        ),

        "soil_units": (
            "OpenLandMap USDA great groups instead of "
            "SoilGrids USDA units"
        ),

        "precipitation_spi": (
            "CHIRPS instead of GPCC "
            "- not available in GEE"
        ),

        "spi_transform": (
            CFG["SPI_METHOD"]
            + " (no incomplete gamma function in Earth Engine)"
        ),

        "population": (
            "WorldPop 2020 age/sex "
            "- later years unavailable"
        ),

        "drought_vulnerability": (
            "rebuilt composite "
            "- JRC DVI not available in GEE"
        ),
    },

    "generated": (
        dt.datetime
        .now(dt.timezone.utc)
        .isoformat(timespec="seconds")
    ),
}



(
    OUT / "provenance.json"
).write_text(
    json.dumps(
        PROVENANCE,
        indent=2,
        ensure_ascii=False,
    )
)


print(
    "wrote",
    len(list(OUT.glob("*"))),
    "files to",
    OUT.resolve(),
)


wrote 7 files to /content/outputs


### I.2 Export des cartes en PNG (remplace l'export GeoTIFF)

Les trois cartes de restitution sont désormais exportées **localement en PNG**, dans le même
dossier `outputs/` que les CSV, avec une légende adaptée à chaque carte — au lieu d'un export
GeoTIFF (asset GEE / Drive).

1. **Carte SDG 15.3.1 (dégradation)** — 3 classes (Dégradé / Stable / Amélioré), palette rouge-jaune-vert.
2. **Carte SPI-12** — indice de sécheresse continu, palette brun-blanc-sarcelle (teintes différentes de la carte de dégradation).
3. **Carte d'alerte de surveillance** — produit *simplifié et statique* (0 Aucune / 1 Vigilance / 2 Alerte / 3 Sévère),
   construit à partir du SPI-12 le plus récent et du statut SDG 15.3.1 déjà calculés plus haut.
   Il ne s'agit **pas** du moteur near-real-time (pas de rafraîchissement en continu, pas de carte
   interactive, pas de service de tuiles MapLibre) — seulement d'une couche d'alerte ponctuelle
   utile pour l'illustration cartographique du rapport.


In [ ]:
import io
import time
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from PIL import Image as PILImage




MAP_EXPORT_DIMENSIONS = 1400



MAP_VIS = {
    "sdg":   {"min": -1, "max": 1, "palette": ["#a50026", "#ffffbf", "#1a9850"]},
    "spi":   {"min": -2.5, "max": 2.5,
              "palette": ["#8c510a", "#f6e8c3", "#f5f5f5", "#c7eae5", "#01665e"]},
    "alert": {"min": 0, "max": 3,
              "palette": ["#f7f7f7", "#fee08b", "#f46d43", "#a50026"]},
}











_MAP_EXPORT_REGION = AOI
_BORDER_IMG = ee.Image().paint(REGIONS, 1, 1.2).visualize(palette=["#ffffff"])


def _fetch_thumbnail(ee_image, vis_params, region, dimensions=MAP_EXPORT_DIMENSIONS,
                      attempts=4, base_timeout=150):
    """
    Download a GEE thumbnail with retry/backoff. Heavy images (e.g. a
    multi-year SPI regression) can be slow to rasterize; on each retry we
    also shrink the requested dimensions a bit, which lightens the
    server-side computation and makes success on a later attempt more
    likely. On an HTTP error, the response body (GEE's actual error message)
    is printed to help diagnose anything that isn't a plain timeout.
    """
    params = dict(vis_params)
    last_err = None
    for i in range(attempts):
        dims = max(600, int(dimensions * (0.8 ** i)))  
        params.update({"region": region, "dimensions": dims, "format": "png"})
        try:
            url = ee_image.getThumbURL(params)
            resp = requests.get(url, timeout=base_timeout + i * 60)
            resp.raise_for_status()
            return PILImage.open(io.BytesIO(resp.content)).convert("RGBA")
        except requests.exceptions.HTTPError as e:
            last_err = e
            body = getattr(e.response, "text", "")
            print(f"  ... getThumbURL attempt {i + 1}/{attempts} failed "
                  f"({e.response.status_code}): {body[:300]}")
        except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError) as e:
            last_err = e
            print(f"  ... getThumbURL attempt {i + 1}/{attempts} timed out")
        if i < attempts - 1:
            wait = 5 * (i + 1)
            print(f"      retrying in {wait}s at dimensions={max(600, int(dimensions * (0.8 ** (i + 1))))}")
            time.sleep(wait)
    raise last_err


def export_static_map(ee_image, vis, title, filename, legend="categorical",
                       legend_items=None, colorbar_label=None,
                       region=None, boundaries=True, out_dir=None):
    """
    Render a GEE image to a PNG map with a proper legend and save it locally
    (outputs/ by default). Legend types:
      - "categorical": legend_items = [(label, hex_color), ...]
      - "continuous":  colorbar built from vis["palette"] / vis["min"] / vis["max"]
    The legend is drawn OUTSIDE the map (to the right) so it never hides data.
    """
    region = region if region is not None else _MAP_EXPORT_REGION
    out_dir = out_dir if out_dir is not None else OUT

    display_img = ee_image.clip(region)

    if boundaries:
        thumb_img = display_img.visualize(**vis).blend(_BORDER_IMG)
        thumb = _fetch_thumbnail(thumb_img, {}, region)
    else:
        thumb = _fetch_thumbnail(display_img, vis, region)

    
    fig, ax = plt.subplots(figsize=(10, 9))
    ax.imshow(thumb)
    ax.axis("off")
    ax.set_title(title, fontsize=14, fontweight="bold", pad=14)

    if legend == "categorical" and legend_items:
        handles = [mpatches.Patch(facecolor=c, edgecolor="#555555", label=l)
                   for l, c in legend_items]
        ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.01, 0.5),
                  frameon=True, fontsize=12, framealpha=0.95, borderpad=1,
                  handlelength=1.6, handleheight=1.6)
        fig.subplots_adjust(right=0.78)
    elif legend == "continuous":
        cmap = LinearSegmentedColormap.from_list("map_cmap", vis["palette"])
        norm = Normalize(vmin=vis["min"], vmax=vis["max"])
        sm = ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.04, pad=0.03)
        cbar.set_label(colorbar_label or "", fontsize=11)
        cbar.ax.tick_params(labelsize=10)

    out_path = Path(out_dir) / filename
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("wrote", out_path)
    return out_path


In [ ]:







def build_alert_static(drought_img, status_img):
    alert = (ee.Image(0)
             .where(drought_img.gte(1), 1)
             .where(drought_img.gte(2), 2)
             .where(drought_img.gte(3).And(status_img.eq(-1)), 3))
    return alert.updateMask(drought_img.mask()).rename("alert").toInt8()

ALERT_MAP = build_alert_static(DROUGHT_2025, STATUS)
print("Static alert layer (ALERT_MAP) ready.")


Static alert layer (ALERT_MAP) ready.


In [ ]:






export_static_map(
    SDG_REPORT, MAP_VIS["sdg"],
    title="SDG 15.3.1 \u2014 \u00c9tat de la d\u00e9gradation des terres ("
          + str(CFG["REPORT_START"]) + "-" + str(CFG["REPORT_END"]) + ")",
    filename="carte_sdg1531_degradation.png",
    legend="categorical",
    legend_items=[("D\u00e9grad\u00e9", "#a50026"), ("Stable", "#ffffbf"), ("Am\u00e9lior\u00e9", "#1a9850")],
)

export_static_map(
    SPI_2025, MAP_VIS["spi"],
    title="SPI-12 \u2014 Indice standardis\u00e9 de pr\u00e9cipitation ("
          + str(CFG["REPORT_END"]) + ")",
    filename="carte_spi12.png",
    legend="continuous",
    colorbar_label="SPI-12 (sec \u2190 \u2192 humide)",
)

try:
    export_static_map(
        ALERT_MAP, MAP_VIS["alert"],
        title="Carte d'alerte de surveillance (SPI-12 & statut de d\u00e9gradation)",
        filename="carte_alerte_surveillance.png",
        legend="categorical",
        legend_items=[("Aucune", "#f7f7f7"), ("Vigilance", "#fee08b"),
                      ("Alerte", "#f46d43"), ("S\u00e9v\u00e8re", "#a50026")],
    )
except Exception as e:
    print(f"WARNING: could not export ALERT_MAP ({e}). It may be fully masked/empty within the AOI.")

print("wrote", len(list(OUT.glob("*.png"))), "PNG map(s) to", OUT.resolve())


wrote outputs/carte_sdg1531_degradation.png
  ... getThumbURL attempt 1/4 timed out
      retrying in 5s at dimensions=1120
wrote outputs/carte_spi12.png
wrote outputs/carte_alerte_surveillance.png
wrote 3 PNG map(s) to /content/outputs


## Validation Checklist Before Anything Leaves the Building

1. **Boundaries**
   - Confirm **12 features** using the 2015 *découpage*.
   - Validate the geometries against HCP's cartographic base.

2. **Trajectory sensitivity**
   - Run both `ndvi_trend` and `restrend`.
   - If the degraded proportion differs by more than a few percentage points, the
     climate signal is dominant.
   - In that case, the uncorrected figure must not be used as the headline result.

3. **State method**
   - Record whether `equal_interval_extended` or `percentile` was used.

4. **Transition matrix**
   - Obtain sign-off from land-degradation specialists.
   - Do not leave the matrix at the default values.

5. **SOC**
   - Acknowledge that SOC is derived from land cover.
   - Check whether the reporting-period length places the `f` transitions above or
     below the ±10% threshold.

6. **MCD12Q1 noise**
   - Inspect grassland ↔ bare-land transitions in the pre-Saharan belt.
   - If they dominate the degraded area, the land-cover sub-indicator may be
     reporting sensor instability rather than actual degradation.

7. **SPI method**
   - State which transformation was used.
   - Document why the GPG gamma-fit method was not used.

8. **Comparability**
   - Run at least one region through the Trends.Earth QGIS plugin.
   - Compare the resulting degraded proportions.
   - Differences are expected because of the substitutions; unexplained
     differences are not.